In [1]:
import os 
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping

In [2]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    '../artifacts/data/processed/train',
    label_mode='int',
    image_size=(224, 224),
    shuffle=True
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    '../artifacts/data/processed/val',
    label_mode='int',
    image_size=(224, 224),
    shuffle=True
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    '../artifacts/data/processed/test',
    label_mode='int',
    image_size=(224, 224),
    shuffle=False
)

Found 10363 files belonging to 2 classes.
Found 3157 files belonging to 2 classes.
Found 3138 files belonging to 2 classes.


In [3]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.2),
    layers.RandomContrast(0.1)
])


In [4]:
base_model = tf.keras.applications.MobileNetV2(
            weights='imagenet',
            input_shape=(224, 224, 3),
            include_top=False
        )

In [ ]:
base_model.trainable = False

num_classes = 2


model = models.Sequential([
    layers.InputLayer(shape=(224, 224, 3)),
    data_augmentation,
    layers.Lambda(tf.keras.applications.mobilenet_v2.preprocess_input),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(64, activation='relu'),
    layers.Dense(num_classes, activation='softmax')
])

d:\SAMITH\Github\Image-Based-Food-Freshness-Prediction-System\venv\Lib\site-packages\keras\src\layers\core\input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


In [6]:
model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
            loss='sparse_categorical_crossentropy',
            metrics=["accuracy"]
        )

In [7]:
early_stopping = EarlyStopping(
            monitor='val_loss',    
            patience=5,
            restore_best_weights=True
        )

In [ ]:
history = model.fit(
            train_ds,
            validation_data=val_ds,
            epochs=50,
            callbacks=[early_stopping]  
        )

Epoch 1/50
266/324 ━━━━━━━━━━━━━━━━━━━━ 56s 976ms/step - accuracy: 0.8522 - loss: 0.3199

In [ ]:

os.makedirs('artifacts/models', exist_ok=True) 
model.save('artifacts/models/with_augmentation.keras')
